# Weather Integration and Demand Analysis

This notebook integrates **TfL Cycle Hire journey demand** with **Open-Meteo historical weather** for London in October 2024.

## Business objective

Extend the bike-share operations analysis by answering:

- How is demand associated with precipitation?
- Does the relationship remain after controlling for hour of day and weekday/weekend patterns?
- How does demand vary across rain intensity and wind conditions?
- Is there a clear temperature relationship in this one-month sample?

Weather relationships in this notebook are **observational associations**, not causal effects. The weather data also represent a single London-area grid point rather than conditions at every docking station.

## 1. Setup and validated journey data

The journey dataset was validated in `01_initial_validation.ipynb`. This notebook uses the October analysis window only.

In [1]:
from pathlib import Path
import json

import pandas as pd
import requests

# Allow the notebook to run from either the project root or notebooks/ folder.
cwd = Path.cwd()
project_dir = cwd.parent if cwd.name == "notebooks" else cwd

processed_dir = project_dir / "data" / "processed"
weather_dir = project_dir / "data" / "raw" / "weather"
weather_dir.mkdir(parents=True, exist_ok=True)

processed_bike_path = processed_dir / "tfl_october_2024_validated.parquet"

# Load the validated October journey dataset.
df_analysis = pd.read_parquet(processed_bike_path)

print("Rows loaded:", len(df_analysis))
print("Start date dtype:", df_analysis["Start date"].dtype)
print("Earliest start:", df_analysis["Start date"].min())
print("Latest start:", df_analysis["Start date"].max())


Rows loaded: 831179
Start date dtype: datetime64[us]
Earliest start: 2024-10-01 00:00:00
Latest start: 2024-10-31 23:59:00


## 2. Retrieve or load historical weather

Open-Meteo's Historical Weather API is queried for a central-London coordinate. The request covers a slightly wider UTC date range so that the complete October period can later be converted correctly to `Europe/London`, including the daylight-saving transition.

For reproducibility, the notebook reuses the saved raw JSON snapshot when it already exists. If it does not exist, the API is queried and the response is saved locally.

In [2]:
# Raw Open-Meteo snapshot used for the analysis.
weather_raw_path = (
    weather_dir / "open_meteo_london_october_2024_utc.json"
)

if weather_raw_path.exists():
    # Reuse the original raw snapshot during reproducible reruns.
    with open(weather_raw_path, "r", encoding="utf-8") as f:
        weather_json = json.load(f)

    print("Loaded existing weather snapshot:", weather_raw_path)

else:
    # Request a wider UTC window so the local October period is complete.
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "start_date": "2024-09-30",
        "end_date": "2024-11-01",
        "hourly": [
            "temperature_2m",
            "precipitation",
            "wind_speed_10m",
            "weather_code",
        ],
        "timezone": "GMT",
        "timeformat": "unixtime",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    weather_json = response.json()

    # Save the raw response so later runs use the same source snapshot.
    with open(weather_raw_path, "w", encoding="utf-8") as f:
        json.dump(weather_json, f, indent=2)

    print("Fetched and saved weather snapshot:", weather_raw_path)

# Inspect the returned grid point and units.
print("Returned latitude:", weather_json["latitude"])
print("Returned longitude:", weather_json["longitude"])
print("Timezone:", weather_json["timezone"])
print("Hourly units:", weather_json["hourly_units"])


Loaded existing weather snapshot: /Users/erik/Documents/Projects/bike-share-operations/data/raw/weather/open_meteo_london_october_2024_utc.json
Returned latitude: 51.493847
Returned longitude: -0.1630249
Timezone: GMT
Hourly units: {'time': 'unixtime', 'temperature_2m': '°C', 'precipitation': 'mm', 'wind_speed_10m': 'km/h', 'weather_code': 'wmo code'}


Open-Meteo returns the center of its weather grid cell rather than necessarily the exact requested coordinate. That is expected and is treated as a city-level weather approximation.

## 3. Convert UTC weather timestamps to London local time

The API response uses Unix timestamps in UTC. Converting from UTC to `Europe/London` preserves the actual daylight-saving transition correctly.

In [3]:
# Convert the hourly API response to a DataFrame.
weather_df = pd.DataFrame(weather_json["hourly"])

weather_df["time_utc"] = pd.to_datetime(
    weather_df["time"],
    unit="s",
    utc=True,
)

weather_df["time_london"] = (
    weather_df["time_utc"]
    .dt.tz_convert("Europe/London")
)

# Keep the exact local October analysis window.
weather_start = pd.Timestamp(
    "2024-10-01 00:00",
    tz="Europe/London",
)

weather_end = pd.Timestamp(
    "2024-11-01 00:00",
    tz="Europe/London",
)

weather_october = weather_df.loc[
    (weather_df["time_london"] >= weather_start)
    & (weather_df["time_london"] < weather_end)
].copy()

weather_columns = [
    "temperature_2m",
    "precipitation",
    "wind_speed_10m",
    "weather_code",
]

print("October weather rows:", len(weather_october))
print("First local timestamp:", weather_october["time_london"].min())
print("Last local timestamp:", weather_october["time_london"].max())
print("Missing weather values:", weather_october[weather_columns].isna().sum().sum())


October weather rows: 745
First local timestamp: 2024-10-01 00:00:00+01:00
Last local timestamp: 2024-10-31 23:00:00+00:00
Missing weather values: 0


October 2024 contains **745 actual elapsed hours** in London because clocks moved back one hour on 27 October. TfL journey timestamps are local, timezone-naive, and minute-level, so the two distinct occurrences of `2024-10-27 01:00` cannot be distinguished in the journey data.

In [4]:
# Create a timezone-naive local-clock key for joining to TfL timestamps.
weather_october["local_hour"] = (
    weather_october["time_london"]
    .dt.tz_localize(None)
    .dt.floor("h")
)

# Identify the duplicated local clock hour caused by the DST rollback.
duplicate_local_hours = weather_october.loc[
    weather_october["local_hour"].duplicated(keep=False),
    ["time_utc", "time_london", "local_hour"],
]

display(duplicate_local_hours)

# Exclude both weather observations for the ambiguous repeated local hour.
weather_hourly_analysis = weather_october.loc[
    ~weather_october["local_hour"].duplicated(keep=False)
].copy()

print("Rows before DST exclusion:", len(weather_october))
print("Rows after DST exclusion:", len(weather_hourly_analysis))
print("Unique usable local hours:", weather_hourly_analysis["local_hour"].nunique())


,time_utc,time_london,local_hour
648,2024-10-27 00:00:00+00:00,2024-10-27 01:00:00+01:00,2024-10-27 01:00:00
649,2024-10-27 01:00:00+00:00,2024-10-27 01:00:00+00:00,2024-10-27 01:00:00


Rows before DST exclusion: 745
Rows after DST exclusion: 743
Unique usable local hours: 743


Both weather observations for the repeated `01:00` hour are excluded from the hourly weather join. This avoids assigning an aggregated TfL local-clock hour to the wrong weather observation.

## 4. Build hourly bike demand and join weather

In [5]:
# Aggregate journey starts to local clock hour.
df_analysis["local_hour"] = (
    df_analysis["Start date"].dt.floor("h")
)

hourly_demand = (
    df_analysis
    .groupby("local_hour")
    .size()
    .reset_index(name="journey_starts")
)

# TfL has one aggregated 01:00 bin on the DST rollback date.
ambiguous_dst_hour = pd.Timestamp("2024-10-27 01:00:00")

hourly_demand_clean = hourly_demand.loc[
    hourly_demand["local_hour"] != ambiguous_dst_hour
].copy()

# Keep only weather fields used in the analysis.
weather_for_merge = weather_hourly_analysis[
    [
        "local_hour",
        "temperature_2m",
        "precipitation",
        "wind_speed_10m",
        "weather_code",
    ]
].copy()

# Join one bike-demand observation to one weather observation per usable hour.
hourly_demand_weather = hourly_demand_clean.merge(
    weather_for_merge,
    on="local_hour",
    how="left",
    validate="one_to_one",
)

print("Hourly bike-demand rows before DST exclusion:", len(hourly_demand))
print("Merged usable hours:", len(hourly_demand_weather))
print("Missing values after merge:", hourly_demand_weather.isna().sum().sum())
print("First hour:", hourly_demand_weather["local_hour"].min())
print("Last hour:", hourly_demand_weather["local_hour"].max())


Hourly bike-demand rows before DST exclusion: 744
Merged usable hours: 743
Missing values after merge: 0
First hour: 2024-10-01 00:00:00
Last hour: 2024-10-31 23:00:00


The final hourly analysis table contains **743 unambiguous local-clock hours** with complete bike-demand and weather observations.

## 5. Precipitation and demand

First, compare wet and dry hours without controlling for time patterns. This raw comparison is descriptive only because wet weather may occur disproportionately at naturally high- or low-demand times.

In [6]:
# Flag hours with any recorded precipitation.
hourly_demand_weather["is_wet"] = (
    hourly_demand_weather["precipitation"] > 0
)

wet_dry_summary = (
    hourly_demand_weather
    .groupby("is_wet")["journey_starts"]
    .agg(
        hour_count="count",
        mean_journey_starts="mean",
        median_journey_starts="median",
    )
    .reset_index()
)

display(wet_dry_summary)

dry_mean = wet_dry_summary.loc[
    ~wet_dry_summary["is_wet"],
    "mean_journey_starts",
].iloc[0]

wet_mean = wet_dry_summary.loc[
    wet_dry_summary["is_wet"],
    "mean_journey_starts",
].iloc[0]

raw_wet_difference_pct = (
    (wet_mean - dry_mean) / dry_mean * 100
)

print("Wet hours:", int(hourly_demand_weather["is_wet"].sum()))
print("Dry hours:", int((~hourly_demand_weather["is_wet"]).sum()))
print("Raw wet vs dry mean difference (%):", round(raw_wet_difference_pct, 2))


,is_wet,hour_count,mean_journey_starts,median_journey_starts
0,False,597,1131.624791,1015.0
1,True,146,1061.856164,1051.0


Wet hours: 146
Dry hours: 597
Raw wet vs dry mean difference (%): -6.17


### Matched dry-hour baseline

To reduce confounding from normal demand patterns, each wet hour is compared with the **average dry demand for the same hour of day and the same weekday/weekend type**.

This is not a causal model, but it is more informative than comparing all wet hours with all dry hours directly.

In [7]:
# Add hour-of-day and weekday/weekend controls.
hourly_demand_weather["hour"] = (
    hourly_demand_weather["local_hour"].dt.hour
)

hourly_demand_weather["is_weekend"] = (
    hourly_demand_weather["local_hour"].dt.dayofweek >= 5
)

# Calculate the typical dry demand for each hour/day-type combination.
dry_baseline = (
    hourly_demand_weather.loc[
        ~hourly_demand_weather["is_wet"]
    ]
    .groupby(["is_weekend", "hour"])["journey_starts"]
    .mean()
    .reset_index(name="expected_dry_journey_starts")
)

# Match every wet hour to its corresponding dry baseline.
wet_hours_comparable = (
    hourly_demand_weather.loc[
        hourly_demand_weather["is_wet"]
    ]
    .merge(
        dry_baseline,
        on=["is_weekend", "hour"],
        how="left",
        validate="many_to_one",
    )
)

# Validate baseline coverage.
baseline_coverage = (
    hourly_demand_weather
    .groupby(["is_weekend", "hour"])["is_wet"]
    .nunique()
)

print(
    "Hour/day-type groups with both wet and dry observations:",
    int((baseline_coverage == 2).sum()),
    "of",
    len(baseline_coverage),
)
print(
    "Wet hours without matching dry baseline:",
    wet_hours_comparable["expected_dry_journey_starts"].isna().sum(),
)

# Aggregate actual and expected demand across comparable wet hours.
actual_wet_starts = wet_hours_comparable["journey_starts"].sum()
expected_dry_starts = (
    wet_hours_comparable["expected_dry_journey_starts"].sum()
)

controlled_wet_difference_pct = (
    (actual_wet_starts - expected_dry_starts)
    / expected_dry_starts
    * 100
)

print("Comparable wet hours:", len(wet_hours_comparable))
print("Actual starts during wet hours:", round(actual_wet_starts))
print("Expected starts from matched dry baseline:", round(expected_dry_starts, 1))
print("Controlled wet vs dry difference (%):", round(controlled_wet_difference_pct, 2))


Hour/day-type groups with both wet and dry observations: 46 of 48
Wet hours without matching dry baseline: 0
Comparable wet hours: 146
Actual starts during wet hours: 155031
Expected starts from matched dry baseline: 188804.9
Controlled wet vs dry difference (%): -17.89


In [8]:
# Compare the matched wet-weather effect for weekdays and weekends.
wet_effect_by_day_type = (
    wet_hours_comparable
    .groupby("is_weekend")
    .agg(
        wet_hours=("journey_starts", "count"),
        actual_starts=("journey_starts", "sum"),
        expected_dry_starts=("expected_dry_journey_starts", "sum"),
    )
    .reset_index()
)

wet_effect_by_day_type["difference_pct"] = (
    (
        wet_effect_by_day_type["actual_starts"]
        - wet_effect_by_day_type["expected_dry_starts"]
    )
    / wet_effect_by_day_type["expected_dry_starts"]
    * 100
)

display(wet_effect_by_day_type)


,is_weekend,wet_hours,actual_starts,expected_dry_starts,difference_pct
0,False,110,123837,146624.774637,-15.541558
1,True,36,31194,42180.095238,-26.045686


In this October sample, the matched comparison shows lower demand during wet hours, with a larger relative reduction on weekends. Weekend trips may be more discretionary, but trip purpose is not observed, so that interpretation remains a hypothesis.

### Rain intensity

The following precipitation bands are **project-specific analytical categories**, not official meteorological classifications:

- **Low:** >0 to 0.2 mm/hour
- **Medium:** >0.2 to 1.5 mm/hour
- **High:** >1.5 mm/hour

In [9]:
# Inspect the positive-precipitation distribution before defining bands.
wet_precipitation = hourly_demand_weather.loc[
    hourly_demand_weather["precipitation"] > 0,
    "precipitation",
]

print(
    wet_precipitation.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

# Assign data-driven precipitation groups to comparable wet hours.
wet_hours_comparable["precipitation_group"] = pd.cut(
    wet_hours_comparable["precipitation"],
    bins=[0, 0.2, 1.5, float("inf")],
    labels=["Low", "Medium", "High"],
    right=True,
)

rain_intensity_summary = (
    wet_hours_comparable
    .groupby("precipitation_group", observed=True)
    .agg(
        wet_hours=("journey_starts", "count"),
        mean_precipitation_mm=("precipitation", "mean"),
        actual_starts=("journey_starts", "sum"),
        expected_dry_starts=("expected_dry_journey_starts", "sum"),
    )
    .reset_index()
)

rain_intensity_summary["difference_pct"] = (
    (
        rain_intensity_summary["actual_starts"]
        - rain_intensity_summary["expected_dry_starts"]
    )
    / rain_intensity_summary["expected_dry_starts"]
    * 100
)

display(rain_intensity_summary)


count    146.000000
mean       0.569863
std        0.915188
min        0.100000
25%        0.100000
50%        0.200000
75%        0.575000
90%        1.450000
95%        2.750000
max        6.100000
Name: precipitation, dtype: float64


,precipitation_group,wet_hours,mean_precipitation_mm,actual_starts,expected_dry_starts,difference_pct
0,Low,84,0.122619,95026,105189.030024,-9.661682
1,Medium,49,0.673469,46545,61022.209418,-23.724492
2,High,13,3.069231,13460,22593.630434,-40.425687


The estimated demand reduction becomes larger as precipitation intensity increases. The **High** group contains only 13 hours, so that estimate is less stable and should be interpreted cautiously.

## 6. Normalize demand for wind and temperature comparisons

For weather variables that vary continuously, demand is normalized against the average demand for the same hour of day and weekday/weekend type:

`demand index = actual starts / typical starts × 100`

A value of **100%** means demand is equal to the typical level for that hour/day type.

In [10]:
# Calculate typical demand for each hour/day-type combination using all conditions.
demand_baseline = (
    hourly_demand_weather
    .groupby(["is_weekend", "hour"])["journey_starts"]
    .mean()
    .reset_index(name="baseline_journey_starts")
)

weather_normalized = hourly_demand_weather.merge(
    demand_baseline,
    on=["is_weekend", "hour"],
    how="left",
    validate="many_to_one",
)

weather_normalized["demand_index_pct"] = (
    weather_normalized["journey_starts"]
    / weather_normalized["baseline_journey_starts"]
    * 100
)

print(weather_normalized["demand_index_pct"].describe())


count    743.000000
mean     100.000000
std       21.412012
min       25.682382
25%       87.858845
50%      102.559856
75%      112.863960
max      190.445860
Name: demand_index_pct, dtype: float64


## 7. Wind speed and normalized demand

Wind bands use approximate October quartile cut points. Because rain and wind can occur together, the dry-only comparison is the more useful check for whether the wind pattern persists without recorded precipitation.

In [11]:
# Group wind speed using approximate October quartile thresholds.
weather_normalized["wind_group"] = pd.cut(
    weather_normalized["wind_speed_10m"],
    bins=[-0.01, 8, 12.6, 17.5, float("inf")],
    labels=["Low", "Moderate", "Elevated", "High"],
)

wind_summary = (
    weather_normalized
    .groupby("wind_group", observed=True)
    .agg(
        hours=("journey_starts", "count"),
        mean_wind_kmh=("wind_speed_10m", "mean"),
        mean_demand_index_pct=("demand_index_pct", "mean"),
        median_demand_index_pct=("demand_index_pct", "median"),
    )
    .reset_index()
)

# Repeat the comparison using dry hours only.
dry_weather_normalized = weather_normalized.loc[
    weather_normalized["precipitation"] == 0
].copy()

dry_wind_summary = (
    dry_weather_normalized
    .groupby("wind_group", observed=True)
    .agg(
        hours=("journey_starts", "count"),
        mean_wind_kmh=("wind_speed_10m", "mean"),
        mean_demand_index_pct=("demand_index_pct", "mean"),
        median_demand_index_pct=("demand_index_pct", "median"),
    )
    .reset_index()
)

print("All weather conditions:")
display(wind_summary)

print("Dry hours only:")
display(dry_wind_summary)


All weather conditions:


,wind_group,hours,mean_wind_kmh,mean_demand_index_pct,median_demand_index_pct
0,Low,192,5.635417,111.266533,109.557050
1,Moderate,182,10.459890,102.030166,105.057377
2,Elevated,184,14.998370,99.026365,102.580246
3,High,185,21.381622,87.278294,91.177956


Dry hours only:


,wind_group,hours,mean_wind_kmh,mean_demand_index_pct,median_demand_index_pct
0,Low,192,5.635417,111.266533,109.557050
1,Moderate,148,10.317568,104.042937,107.972087
2,Elevated,138,14.821739,102.088267,104.141900
3,High,119,20.904202,92.649365,95.384293


Normalized demand declines across the wind-speed groups, and the pattern remains in the dry-only subset. This supports an association between higher wind speeds and lower bike demand in this sample, without establishing causality.

## 8. Temperature and normalized demand

Temperature is divided into sample quartiles. A dry-only comparison is also included to reduce precipitation as a confounding factor.

In [12]:
# Divide October temperatures into four equally sized sample groups.
weather_normalized["temperature_group"] = pd.qcut(
    weather_normalized["temperature_2m"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
)

temperature_ranges = (
    weather_normalized
    .groupby("temperature_group", observed=True)
    .agg(
        hours=("journey_starts", "count"),
        min_temperature=("temperature_2m", "min"),
        max_temperature=("temperature_2m", "max"),
        mean_temperature=("temperature_2m", "mean"),
    )
    .reset_index()
)

temperature_summary = (
    weather_normalized
    .groupby("temperature_group", observed=True)
    .agg(
        mean_demand_index_pct=("demand_index_pct", "mean"),
        median_demand_index_pct=("demand_index_pct", "median"),
    )
    .reset_index()
)

dry_temperature_summary = (
    weather_normalized.loc[
        weather_normalized["precipitation"] == 0
    ]
    .groupby("temperature_group", observed=True)
    .agg(
        hours=("journey_starts", "count"),
        mean_temperature=("temperature_2m", "mean"),
        mean_demand_index_pct=("demand_index_pct", "mean"),
        median_demand_index_pct=("demand_index_pct", "median"),
    )
    .reset_index()
)

print("Temperature ranges:")
display(temperature_ranges)

print("All weather conditions:")
display(temperature_summary)

print("Dry hours only:")
display(dry_temperature_summary)


Temperature ranges:


,temperature_group,hours,min_temperature,max_temperature,mean_temperature
0,Q1,188,1.9,10.1,8.073404
1,Q2,190,10.2,12.4,11.341579
2,Q3,182,12.5,14.4,13.481319
3,Q4,183,14.5,21.4,15.851366


All weather conditions:


,temperature_group,mean_demand_index_pct,median_demand_index_pct
0,Q1,103.130877,103.109855
1,Q2,104.426325,104.700382
2,Q3,93.575828,98.231514
3,Q4,98.577010,102.723969


Dry hours only:


,temperature_group,hours,mean_temperature,mean_demand_index_pct,median_demand_index_pct
0,Q1,171,7.969591,105.245571,104.408919
1,Q2,174,11.324138,105.704838,105.894813
2,Q3,127,13.448819,98.718007,102.710962
3,Q4,125,15.824800,103.585303,105.802964


There is no clear monotonic temperature-demand relationship in this one-month October sample, so temperature is not emphasized in the final dashboard.

## 9. Export processed datasets

During QA, the notebook writes to `_test.parquet` filenames so the existing validated outputs are not overwritten. After the new files are verified against the originals, these paths can be changed back to the production filenames.

In [ ]:
# QA output: merged hourly bike-demand and weather table.
processed_weather_path = (
    processed_dir / "hourly_demand_weather_october_2024.parquet"
)

hourly_demand_weather.to_parquet(
    processed_weather_path,
    index=False,
)

# Reload the file to confirm it was written successfully.
weather_check = pd.read_parquet(processed_weather_path)

print("Saved:", processed_weather_path)
print("Rows:", len(weather_check))
print("Columns:", weather_check.columns.tolist())
print("Missing values:", weather_check.isna().sum().sum())


Saved: /Users/erik/Documents/Projects/bike-share-operations/data/processed/hourly_demand_weather_october_2024_test.parquet
Rows: 743
Columns: ['local_hour', 'journey_starts', 'temperature_2m', 'precipitation', 'wind_speed_10m', 'weather_code', 'is_wet', 'hour', 'is_weekend']
Missing values: 0


In [ ]:
# Create the compact warehouse-ready weather table.
weather_bigquery = weather_hourly_analysis[
    [
        "local_hour",
        "temperature_2m",
        "precipitation",
        "wind_speed_10m",
        "weather_code",
    ]
].copy()

weather_bigquery = weather_bigquery.rename(
    columns={
        "temperature_2m": "temperature_c",
        "precipitation": "precipitation_mm",
        "wind_speed_10m": "wind_speed_kmh",
    }
)

# QA output: keep the existing BigQuery-ready file untouched until verified.
weather_bq_path = (
    processed_dir / "weather_hourly_october_2024_bigquery.parquet"
)

weather_bigquery.to_parquet(
    weather_bq_path,
    index=False,
)

print("Saved:", weather_bq_path)
print("Rows:", len(weather_bigquery))
print("Columns:", weather_bigquery.columns.tolist())
print("Duplicate local hours:", weather_bigquery["local_hour"].duplicated().sum())
print("Missing values:", weather_bigquery.isna().sum().sum())


Saved: /Users/erik/Documents/Projects/bike-share-operations/data/processed/weather_hourly_october_2024_bigquery_test.parquet
Rows: 743
Columns: ['local_hour', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh', 'weather_code']
Duplicate local hours: 0
Missing values: 0


## Key findings and analytical decisions

- Weather was converted from UTC to `Europe/London` before creating the local-hour join key.
- The repeated `2024-10-27 01:00` DST hour was excluded from the weather-demand join because TfL's timezone-naive timestamps cannot distinguish its two occurrences.
- The final merged dataset contains 743 complete, unambiguous hourly observations.
- Wet-hour demand was evaluated primarily against a matched dry baseline for the same hour and weekday/weekend type rather than through the raw wet-vs-dry comparison.
- Demand was lower during wet conditions in the matched comparison, with larger reductions at higher precipitation intensities.
- Higher wind speed was associated with lower normalized demand, including when restricted to dry hours.
- No clear directional temperature relationship was found in this October-only sample.
- Weather relationships are observational and are based on a single London-area weather grid point.